In [1]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command, interrupt
from rich import print


# 声明状态
class State(TypedDict):
    username: str
    age: str


# 声明节点
def ask_name(state: State) -> dict:
    print("=========== 执行 ask_name 节点===========")  # 被中断的任务会重放：横幅打印两次（暂停前 + 恢复重跑）
    username = interrupt("请输入你的名字")
    return {
        "username": username,
    }


def ask_age(state: State) -> dict:
    print("=========== 执行 ask_age 节点===========")  # 未被中断的任务不重放：横幅只打印一次（本轮已完成并提交）
    # age = interrupt("请输入你的年龄")
    return {
        "age": 26,
    }


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("ask_name", ask_name)
builder.add_node("ask_age", ask_age)
builder.add_edge(START, "ask_name")
builder.add_edge(START, "ask_age")
builder.add_edge("ask_name", END)
builder.add_edge("ask_age", END)

# 使用中断必须设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)

=========== 执行 ask_name 节点===========

=========== 执行 ask_age 节点===========

{'age': 26, '__interrupt__': [Interrupt(value='请输入你的名字', id='357113b375d838e2c3d2ac6553d9e4dc')]}

In [2]:
# 恢复执行
resume_map = {}  # { id : msg }
for i in res['__interrupt__']:
    ask_msg = input(f"{i.value}")
    resume_map[i.id] = ask_msg
resume_res = graph.invoke(Command(resume=resume_map), config=config)
print(resume_res)

=========== 执行 ask_name 节点===========

{'username': 'aihaipeng', 'age': 26}

In [3]:
print(list(graph.get_state_history(config=config)))

[
    StateSnapshot(
        values={'username': 'aihaipeng', 'age': 26},
        next=(),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1b122c-c83e-6a7c-8001-56592f2f6a43'
            }
        },
        metadata={'source': 'loop', 'step': 1, 'parents': {}},
        created_at='2026-09-15T16:30:44.339767+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1b122b-4e96-6eba-8000-391c1e992b1b'
            }
        },
        tasks=(),
        interrupts=()
    ),
    StateSnapshot(
        values={},
        next=('ask_name', 'ask_age'),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1b122b-4e96-6eba-8000-391c1e992b1b'
            }
        },
        metadata={'source': 'loop', 'step': 0, 'parents': {}},
        created_at='2026-09-15T16:30:04.739747+00:00',
        parent_config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1b122b-4e95-63db-bfff-1ce1943c30ac'
            }
        },
        tasks=(
            PregelTask(
                id='f94cc43b-f48c-b414-c344-e70c6baa4075',
                name='ask_name',
                path=('__pregel_pull', 'ask_name'),
                error=None,
                interrupts=(Interrupt(value='请输入你的名字', id='357113b375d838e2c3d2ac6553d9e4dc'),),
                state=None,
                result={'username': 'aihaipeng'}
            ),
            PregelTask(
                id='6a7c553c-27ad-06d7-d5ee-c6e246bec0cb',
                name='ask_age',
                path=('__pregel_pull', 'ask_age'),
                error=None,
                interrupts=(),
                state=None,
                result={'age': 26}
            )
        ),
        interrupts=(Interrupt(value='请输入你的名字', id='357113b375d838e2c3d2ac6553d9e4dc'),)
    ),
    StateSnapshot(
        values={},
        next=('__start__',),
        config={
            'configurable': {
                'thread_id': '123',
                'checkpoint_ns': '',
                'checkpoint_id': '1f1b122b-4e95-63db-bfff-1ce1943c30ac'
            }
        },
        metadata={'source': 'input', 'step': -1, 'parents': {}},
        created_at='2026-09-15T16:30:04.739063+00:00',
        parent_config=None,
        tasks=(
            PregelTask(
                id='b614b832-2051-3c84-9d20-f40feb25dfff',
                name='__start__',
                path=('__pregel_pull', '__start__'),
                error=None,
                interrupts=(),
                state=None,
                result={}
            ),
        ),
        interrupts=()
    )
]